In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go

print("Notebook environment is ready.")
print("Pandas version:", pd.__version__)
print("Plotly version:", plotly.__version__)
print("DuckDB version:", duckdb.__version__)

Notebook environment is ready.
Pandas version: 3.0.5
Plotly version: 6.9.0
DuckDB version: 1.5.5


In [2]:
from pathlib import Path

# Works whether Jupyter starts from the project root or notebooks folder
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

INKAR_FOLDER = PROJECT_ROOT / "data" / "raw" / "inkar" / "extracted"

CSV_PATH = INKAR_FOLDER / "inkar_2025.csv"
INDICATOR_PATH = INKAR_FOLDER / "Indikatorenübersicht (INKAR 2025).xlsx"
REFERENCE_PATH = INKAR_FOLDER / "BBSR_Raumgliederungen_Referenz_2023.xlsx"

print("Project root:", PROJECT_ROOT)
print("CSV exists:", CSV_PATH.exists())
print("Indicator file exists:", INDICATOR_PATH.exists())
print("Reference file exists:", REFERENCE_PATH.exists())

Project root: C:\Users\maazm\Desktop\Projects\datacareer-germany
CSV exists: True
Indicator file exists: True
Reference file exists: True


In [3]:
indicator_excel = pd.ExcelFile(INDICATOR_PATH)

print("Indicator workbook sheets:")
for sheet in indicator_excel.sheet_names:
    print("-", sheet)

Indicator workbook sheets:
- Nutzungshinweise
- Raumbeobachtung DE
- ZOM
- SDG
- Raumbeobachtung EU


C:\Users\maazm\Desktop\Projects\datacareer-germany\.venv\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Raumbeobachtung DE'!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


In [4]:
indicator_raw = pd.read_excel(
    INDICATOR_PATH,
    sheet_name="Raumbeobachtung DE",
    header=None
)

print("Shape:", indicator_raw.shape)
display(indicator_raw.head(20))

Shape: (459, 9)


,0,1,2,3,4,5,6,7,8
0,INKAR 2025 – Indikatorenübersicht: Raumbeoba...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kurzname,Name,Algorithmus,M_ID,Kürzel,Anmerkungen,Statistische Grundlagen,Gemeinden,Kreise
2,Absolutzahlen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bodenfläche gesamt qkm,Katasterfläche in km²,NaN,1,TN23-kataster_qkm,Als Katasterfläche bezeichnet man den vermessu...,Laufende Raumbeobachtung des BBSR; Flächenerhe...,2016-2023,2016-2023
4,Bevölkerung gesamt,Zahl der Einwohner insgesamt,NaN,2,xbev,Es handelt sich um die Zahl der Einwohner zum ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
5,Bevölkerung männlich,Zahl der männlichen Einwohner,NaN,3,xbevm,Es handelt sich um die Zahl der männlichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
6,Bevölkerung weiblich,Zahl der weiblichen Einwohner,NaN,4,xbevf,Es handelt sich um die Zahl der weiblichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
7,Bevölkerung (mit BBSR-Zensuskorrekturen),Zensuskorrigierte Zahl der Einwohner insgesamt,NaN,5,bev_korr,Zwischen der Zahl der Bevölkerung am 31.12. ei...,Laufende Raumbeobachtung des BBSR,1995-2023,1995-2023
8,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Zahl der Einwohner im Alter von 15 bis unter 6...,NaN,6,ewf_1565_ges,Es handelt sich um die Zahl der Einwohner und ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,2001-2023,1995-2023
9,Erwerbstätige,Zahl der Erwerbstätigen in 1000 Personen,NaN,7,et1000,Es handelt sich um die Zahl der Erwerbstätigen...,Laufende Raumbeobachtung des BBSR; Arbeitskrei...,NaN,2000-2022


In [5]:
indicator_df = pd.read_excel(
    INDICATOR_PATH,
    sheet_name="Raumbeobachtung DE",
    header=1
)

print("Shape:", indicator_df.shape)
print("Columns:")
print(indicator_df.columns.tolist())

display(indicator_df.head(10))

Shape: (457, 9)
Columns:
['Kurzname', 'Name', 'Algorithmus', 'M_ID', 'Kürzel', 'Anmerkungen', 'Statistische Grundlagen', 'Gemeinden', 'Kreise']


C:\Users\maazm\Desktop\Projects\datacareer-germany\.venv\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Raumbeobachtung DE'!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


,Kurzname,Name,Algorithmus,M_ID,Kürzel,Anmerkungen,Statistische Grundlagen,Gemeinden,Kreise
0,Absolutzahlen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bodenfläche gesamt qkm,Katasterfläche in km²,NaN,1.0,TN23-kataster_qkm,Als Katasterfläche bezeichnet man den vermessu...,Laufende Raumbeobachtung des BBSR; Flächenerhe...,2016-2023,2016-2023
2,Bevölkerung gesamt,Zahl der Einwohner insgesamt,NaN,2.0,xbev,Es handelt sich um die Zahl der Einwohner zum ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
3,Bevölkerung männlich,Zahl der männlichen Einwohner,NaN,3.0,xbevm,Es handelt sich um die Zahl der männlichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
4,Bevölkerung weiblich,Zahl der weiblichen Einwohner,NaN,4.0,xbevf,Es handelt sich um die Zahl der weiblichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
5,Bevölkerung (mit BBSR-Zensuskorrekturen),Zensuskorrigierte Zahl der Einwohner insgesamt,NaN,5.0,bev_korr,Zwischen der Zahl der Bevölkerung am 31.12. ei...,Laufende Raumbeobachtung des BBSR,1995-2023,1995-2023
6,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Zahl der Einwohner im Alter von 15 bis unter 6...,NaN,6.0,ewf_1565_ges,Es handelt sich um die Zahl der Einwohner und ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,2001-2023,1995-2023
7,Erwerbstätige,Zahl der Erwerbstätigen in 1000 Personen,NaN,7.0,et1000,Es handelt sich um die Zahl der Erwerbstätigen...,Laufende Raumbeobachtung des BBSR; Arbeitskrei...,NaN,2000-2022
8,Sozialversicherungspflichtig Beschäftigte am A...,Zahl der sozialversicherungspflichtig Beschäft...,NaN,8.0,sva,Es handelt sich um die Zahl der sozialversiche...,Laufende Raumbeobachtung des BBSR; Beschäftigt...,1997-2023,1997-2023
9,Sozialversicherungspflichtig Beschäftigte am W...,Zahl der sozialversicherungspflichtig Beschäft...,NaN,9.0,svw,Es handelt sich um die Zahl der sozialversiche...,Laufende Raumbeobachtung des BBSR; Beschäftigt...,1997-2023,1997-2023


In [6]:
valid_indicators = indicator_df[
    indicator_df["M_ID"].notna()
    & indicator_df["Kürzel"].notna()
].copy()

valid_indicators["M_ID"] = (
    pd.to_numeric(valid_indicators["M_ID"], errors="coerce")
    .astype("Int64")
)

valid_indicators = valid_indicators[
    [
        "Kurzname",
        "Name",
        "M_ID",
        "Kürzel",
        "Anmerkungen",
        "Gemeinden",
        "Kreise"
    ]
].reset_index(drop=True)

print("Number of valid indicators:", len(valid_indicators))

display(valid_indicators.head(10))

Number of valid indicators: 413


,Kurzname,Name,M_ID,Kürzel,Anmerkungen,Gemeinden,Kreise
0,Bodenfläche gesamt qkm,Katasterfläche in km²,1,TN23-kataster_qkm,Als Katasterfläche bezeichnet man den vermessu...,2016-2023,2016-2023
1,Bevölkerung gesamt,Zahl der Einwohner insgesamt,2,xbev,Es handelt sich um die Zahl der Einwohner zum ...,1995-2023,1995-2023
2,Bevölkerung männlich,Zahl der männlichen Einwohner,3,xbevm,Es handelt sich um die Zahl der männlichen Ein...,1995-2023,1995-2023
3,Bevölkerung weiblich,Zahl der weiblichen Einwohner,4,xbevf,Es handelt sich um die Zahl der weiblichen Ein...,1995-2023,1995-2023
4,Bevölkerung (mit BBSR-Zensuskorrekturen),Zensuskorrigierte Zahl der Einwohner insgesamt,5,bev_korr,Zwischen der Zahl der Bevölkerung am 31.12. ei...,1995-2023,1995-2023
5,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Zahl der Einwohner im Alter von 15 bis unter 6...,6,ewf_1565_ges,Es handelt sich um die Zahl der Einwohner und ...,2001-2023,1995-2023
6,Erwerbstätige,Zahl der Erwerbstätigen in 1000 Personen,7,et1000,Es handelt sich um die Zahl der Erwerbstätigen...,NaN,2000-2022
7,Sozialversicherungspflichtig Beschäftigte am A...,Zahl der sozialversicherungspflichtig Beschäft...,8,sva,Es handelt sich um die Zahl der sozialversiche...,1997-2023,1997-2023
8,Sozialversicherungspflichtig Beschäftigte am W...,Zahl der sozialversicherungspflichtig Beschäft...,9,svw,Es handelt sich um die Zahl der sozialversiche...,1997-2023,1997-2023
9,Arbeitslose,Zahl der Arbeitslosen insgesamt,10,alo,Es handelt sich um die Zahl der Arbeitslosen i...,1998-2023,1995-2023


In [7]:
search_terms = [
    "Arbeitslosenquote",
    "Bevölkerung",
    "Beschäftigte",
    "Erwerbstätige",
    "Bruttoinlandsprodukt",
    "Breitband",
    "Miete",
    "Wanderung",
    "wissensintensiv"
]

pattern = "|".join(search_terms)

indicator_candidates = valid_indicators[
    valid_indicators["Kurzname"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print("Matching indicators:", len(indicator_candidates))

display(
    indicator_candidates[
        [
            "Kurzname",
            "M_ID",
            "Kürzel",
            "Gemeinden",
            "Kreise"
        ]
    ].head(100)
)

Matching indicators: 76


,Kurzname,M_ID,Kürzel,Gemeinden,Kreise
1,Bevölkerung gesamt,2,xbev,1995-2023,1995-2023
2,Bevölkerung männlich,3,xbevm,1995-2023,1995-2023
3,Bevölkerung weiblich,4,xbevf,1995-2023,1995-2023
4,Bevölkerung (mit BBSR-Zensuskorrekturen),5,bev_korr,1995-2023,1995-2023
5,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),6,ewf_1565_ges,2001-2023,1995-2023
...,...,...,...,...,...
398,Bruttoinlandsprodukt je Erwerbstätigen,14206,q_bip_et,NaN,1992-2022
399,Bruttowertschöpfung je Erwerbstätigen,14207,q_bws_et,NaN,2000-2022
400,Bruttowertschöpfung je Erwerbstätigen Primärer...,14208,q_bws_1sektor,NaN,2000-2022
401,Bruttowertschöpfung je Erwerbstätigen Sekundär...,14209,q_bws_2sektor,NaN,2000-2022


In [29]:
def find_indicators(*terms):
    """
    Search INKAR indicators across the short name,
    full name, and notes columns.
    """
    searchable_columns = [
        "Kurzname",
        "Name",
        "Anmerkungen"
    ]

    pattern = "|".join(terms)

    mask = False

    for column in searchable_columns:
        mask = mask | valid_indicators[column].astype(str).str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )

    results = valid_indicators.loc[
        mask,
        [
            "Kurzname",
            "Name",
            "M_ID",
            "Kürzel",
            "Gemeinden",
            "Kreise"
        ]
    ].drop_duplicates()

    return results.reset_index(drop=True)

In [30]:
required_candidates = find_indicators(
    "Arbeitslosenquote",
    "Beschäftigtenentwicklung",
    "Beschäftigte",
    "Erwerbstätige",
    "Bruttoinlandsprodukt",
    "Wanderungssaldo",
    "Breitband",
    "Miete",
    "Mietpreis",
    "wissensintensiv",
    "hochqualifiziert",
    "Bevölkerungsentwicklung"
)

print(
    required_candidates.to_string(
        index=False,
        columns=[
            "Kurzname",
            "M_ID",
            "Kürzel",
            "Gemeinden",
            "Kreise"
        ]
    )
)

                                                              Kurzname  M_ID                       Kürzel Gemeinden    Kreise
                                                         Erwerbstätige     7                       et1000       NaN 2000-2022
               Sozialversicherungspflichtig Beschäftigte am Arbeitsort     8                          sva 1997-2023 1997-2023
                  Sozialversicherungspflichtig Beschäftigte am Wohnort     9                          svw 1997-2023 1997-2023
                                     Bruttoinlandsprodukt in 1000 Euro    11                          bip       NaN 2000-2022
                                                     Arbeitslosenquote  1101                        q_alo       NaN 1998-2023
                                              Arbeitslosenquote Frauen  1102                      q_alo_f       NaN 2008-2023
                                              Arbeitslosenquote Männer  1103                      q_alo_m       NaN 20

In [31]:
candidate_output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "indicator_candidates.csv"
)

required_candidates.to_csv(
    candidate_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:", candidate_output_path)
print("File exists:", candidate_output_path.exists())

Saved to: C:\Users\maazm\Desktop\Projects\datacareer-germany\data\processed\indicator_candidates.csv
File exists: True


In [32]:
SELECTED_CODES = {
    "xbev": "population",
    "q_alo": "unemployment_rate",
    "m_mietpr": "asking_rent",
    "q_svw": "employment_rate",
    "a_sva_exp": "expert_level_employment",
    "a_svb_wissen": "knowledge_intensive_industry",
    "a_svb_IT": "it_science_service_employment",
    "e10_bev": "population_change_10_years",
    "i_wans": "total_migration_balance",
    "m_ek_akad": "median_income_academic",
    "a_bb_100Mbits": "broadband_100mbit",
    "q_bip_ew": "gdp_per_capita"
}

selected_indicators = valid_indicators[
    valid_indicators["Kürzel"].isin(SELECTED_CODES.keys())
].copy()

selected_indicators["project_column"] = selected_indicators[
    "Kürzel"
].map(SELECTED_CODES)

selected_indicators = selected_indicators[
    [
        "project_column",
        "Kurzname",
        "M_ID",
        "Kürzel",
        "Gemeinden",
        "Kreise"
    ]
].sort_values("project_column").reset_index(drop=True)

display(selected_indicators)

,project_column,Kurzname,M_ID,Kürzel,Gemeinden,Kreise
0,asking_rent,Angebotsmietpreise,2113,m_mietpr,NaN,2010-2024
1,broadband_100mbit,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,13117,a_bb_100Mbits,2017-2023,2017-2023
2,employment_rate,Beschäftigtenquote,3101,q_svw,2001-2023,1997-2023
3,expert_level_employment,Beschäftigte mit Anforderungsniveau Experte,3207,a_sva_exp,NaN,2013-2023
4,gdp_per_capita,Bruttoinlandsprodukt je Einwohner,14205,q_bip_ew,NaN,1992-2022
5,it_science_service_employment,Beschäftigte in IT- und naturwissenschaftliche...,3409,a_svb_IT,NaN,2013-2023
6,knowledge_intensive_industry,Beschäftigte in wissensintensiven Industrien,3408,a_svb_wissen,NaN,2009-2023
7,median_income_academic,Medianeinkommen akademischer Berufsabschluss,6009,m_ek_akad,NaN,2014-2023
8,population,Bevölkerung gesamt,2,xbev,1995-2023,1995-2023
9,population_change_10_years,Bevölkerungsentwicklung (10 Jahre),4217,e10_bev,NaN,2005-2023


In [33]:
found_codes = set(selected_indicators["Kürzel"])
missing_codes = set(SELECTED_CODES) - found_codes

print("Indicators selected:", len(selected_indicators))
print("Missing codes:", missing_codes)

Indicators selected: 12
Missing codes: set()


In [34]:
selected_indicator_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "selected_inkar_indicators.csv"
)

selected_indicators.to_csv(
    selected_indicator_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", selected_indicator_path)

Saved: C:\Users\maazm\Desktop\Projects\datacareer-germany\data\processed\selected_inkar_indicators.csv


In [35]:
import csv

with open(
    CSV_PATH,
    "r",
    encoding="utf-8-sig",
    errors="replace",
    newline=""
) as file:
    csv_sample_text = file.read(100_000)

dialect = csv.Sniffer().sniff(
    csv_sample_text,
    delimiters=";,\t|"
)

CSV_DELIMITER = dialect.delimiter

print("Detected delimiter:", repr(CSV_DELIMITER))

Detected delimiter: ';'


In [36]:
csv_preview = pd.read_csv(
    CSV_PATH,
    sep=CSV_DELIMITER,
    nrows=5,
    dtype=str,
    encoding="utf-8-sig",
    encoding_errors="replace"
)

print("Number of columns:", len(csv_preview.columns))

print("\nColumn names:")
for column in csv_preview.columns:
    print("-", column)

display(csv_preview)

Number of columns: 9

Column names:
- Bereich
- ID
- Kuerzel
- Indikator
- Raumbezug
- Kennziffer
- Name
- Zeitbezug
- Wert


,Bereich,ID,Kuerzel,Indikator,Raumbezug,Kennziffer,Name,Zeitbezug,Wert
0,EU,18668,q_alo,Arbeitslosenquote,EU27,EU27,Europäische Union - 27 Länder (ab 2020),2021,"7,00"
1,EU,18668,q_alo,Arbeitslosenquote,NUTS0,AT,Österreich,2021,"6,20"
2,EU,18668,q_alo,Arbeitslosenquote,NUTS0,BE,Belgien,2021,"6,30"
3,EU,18668,q_alo,Arbeitslosenquote,NUTS0,BG,Bulgarien,2021,"5,30"
4,EU,18668,q_alo,Arbeitslosenquote,NUTS0,CY,Zypern,2021,"7,50"


In [37]:
DUCKDB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "datacareer_germany.duckdb"
)

connection = duckdb.connect(str(DUCKDB_PATH))

print("DuckDB database:", DUCKDB_PATH)

DuckDB database: C:\Users\maazm\Desktop\Projects\datacareer-germany\data\processed\datacareer_germany.duckdb


In [38]:
# Convert the Windows path into a format DuckDB can read
csv_sql_path = CSV_PATH.as_posix().replace("'", "''")

connection.execute(f"""
    CREATE OR REPLACE VIEW inkar_raw AS
    SELECT *
    FROM read_csv_auto(
        '{csv_sql_path}',
        delim=';',
        header=True,
        all_varchar=True
    )
""")

print("INKAR CSV view created successfully.")

INKAR CSV view created successfully.


In [39]:
test_rows = connection.execute("""
    SELECT *
    FROM inkar_raw
    LIMIT 5
""").df()

display(test_rows)

,Bereich,ID,Kuerzel,Indikator,Raumbezug,Kennziffer,Name,Zeitbezug,Wert
0,EU,18668,q_alo,Arbeitslosenquote,EU27,EU27,Europäische Union - 27 Länder (ab 2020),2021,"7,00"
1,EU,18668,q_alo,Arbeitslosenquote,NUTS0,AT,Österreich,2021,"6,20"
2,EU,18668,q_alo,Arbeitslosenquote,NUTS0,BE,Belgien,2021,"6,30"
3,EU,18668,q_alo,Arbeitslosenquote,NUTS0,BG,Bulgarien,2021,"5,30"
4,EU,18668,q_alo,Arbeitslosenquote,NUTS0,CY,Zypern,2021,"7,50"


In [40]:
selected_codes_sql = ", ".join(
    f"'{code}'"
    for code in SELECTED_CODES.keys()
)

geo_summary_query = f"""
    SELECT
        Bereich,
        Raumbezug,
        COUNT(*) AS row_count,
        COUNT(DISTINCT Kennziffer) AS number_of_regions,
        MIN(TRY_CAST(Zeitbezug AS INTEGER)) AS earliest_year,
        MAX(TRY_CAST(Zeitbezug AS INTEGER)) AS latest_year
    FROM inkar_raw
    WHERE Kuerzel IN ({selected_codes_sql})
    GROUP BY
        Bereich,
        Raumbezug
    ORDER BY
        Bereich,
        Raumbezug
"""

geo_summary = connection.execute(
    geo_summary_query
).df()

display(geo_summary)

,Bereich,Raumbezug,row_count,number_of_regions,earliest_year,latest_year
0,EU,EU27,2,1,2021,2021
1,EU,NUTS0,54,27,2021,2021
2,EU,NUTS1,183,92,2021,2021
3,EU,NUTS2,480,242,2021,2021
4,LRB,Arbeitsmarktregionen,49042,223,1995,2023
...,...,...,...,...,...,...
79,ZOM,Zentralörtliche Einstufung (zusammengefasst),48,4,2017,2023
80,ZOM,Zentralörtliche Funktion - Grundzentrum,30168,2514,2017,2023
81,ZOM,Zentralörtliche Funktion - Mittelzentrum,11532,961,2017,2023
82,ZOM,Zentralörtliche Funktion - Oberzentrum,1836,153,2017,2023


In [41]:
state_level_candidates = geo_summary[
    geo_summary["number_of_regions"].between(15, 17)
].copy()

display(state_level_candidates)

,Bereich,Raumbezug,row_count,number_of_regions,earliest_year,latest_year
8,LRB,Bundesländer,3760,16,1995,2024
16,LRB,Metropolregionen,3300,15,1995,2023
21,LRB,Regionalstatistischer Raumtyp 17 (RegioStaR17),1785,17,1995,2023
43,SDG,Bundesländer,1200,16,1995,2024
51,SDG,Metropolregionen,900,15,1995,2023
56,SDG,Regionalstatistischer Raumtyp 17 (RegioStaR17),476,17,2001,2023
75,ZOM,Bundesländer,192,16,2017,2023


In [42]:
state_raw_query = f"""
    SELECT
        Bereich,
        ID,
        Kuerzel,
        Indikator,
        Raumbezug,
        Kennziffer,
        Name,
        Zeitbezug,
        Wert
    FROM inkar_raw
    WHERE Bereich = 'LRB'
      AND Raumbezug = 'Bundesländer'
      AND Kuerzel IN ({selected_codes_sql})
    ORDER BY
        Kennziffer,
        Kuerzel,
        TRY_CAST(Zeitbezug AS INTEGER)
"""

state_raw = connection.execute(state_raw_query).df()

print("Shape:", state_raw.shape)
print("Number of states:", state_raw["Kennziffer"].nunique())
print("Number of indicators:", state_raw["Kuerzel"].nunique())

display(state_raw.head(20))

Shape: (3760, 9)
Number of states: 16
Number of indicators: 12


,Bereich,ID,Kuerzel,Indikator,Raumbezug,Kennziffer,Name,Zeitbezug,Wert
0,LRB,17274,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,Bundesländer,01,Schleswig-Holstein,2017,"75,61"
1,LRB,17274,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,Bundesländer,01,Schleswig-Holstein,2020,"89,94"
2,LRB,17274,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,Bundesländer,01,Schleswig-Holstein,2021,"90,31"
3,LRB,17274,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,Bundesländer,01,Schleswig-Holstein,2022,"92,83"
4,LRB,17274,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,Bundesländer,01,Schleswig-Holstein,2023,"95,32"
5,LRB,17524,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,Bundesländer,01,Schleswig-Holstein,2013,"9,58"
6,LRB,17524,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,Bundesländer,01,Schleswig-Holstein,2014,"9,66"
7,LRB,17524,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,Bundesländer,01,Schleswig-Holstein,2015,"9,81"
8,LRB,17524,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,Bundesländer,01,Schleswig-Holstein,2016,"10,07"
9,LRB,17524,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,Bundesländer,01,Schleswig-Holstein,2017,"10,20"


In [43]:
state_names = (
    state_raw[
        ["Kennziffer", "Name"]
    ]
    .drop_duplicates()
    .sort_values("Kennziffer")
    .reset_index(drop=True)
)

print(state_names.to_string(index=False))

Kennziffer                   Name
        01     Schleswig-Holstein
        02                Hamburg
        03          Niedersachsen
        04                 Bremen
        05    Nordrhein-Westfalen
        06                 Hessen
        07        Rheinland-Pfalz
        08      Baden-Württemberg
        09                 Bayern
        10               Saarland
        11                 Berlin
        12            Brandenburg
        13 Mecklenburg-Vorpommern
        14                Sachsen
        15         Sachsen-Anhalt
        16              Thüringen


In [44]:
extracted_indicator_check = (
    state_raw[
        ["Kuerzel", "Indikator"]
    ]
    .drop_duplicates()
    .sort_values("Kuerzel")
    .reset_index(drop=True)
)

print(extracted_indicator_check.to_string(index=False))

      Kuerzel                                                              Indikator
a_bb_100Mbits                         Bandbreitenverfügbarkeit mindestens 100 Mbit/s
    a_sva_exp                            Beschäftigte mit Anforderungsniveau Experte
     a_svb_IT Beschäftigte in IT- und naturwissenschaftlichen Dienstleistungsberufen
 a_svb_wissen                           Beschäftigte in wissensintensiven Industrien
      e10_bev                                     Bevölkerungsentwicklung (10 Jahre)
       i_wans                                                  Gesamtwanderungssaldo
    m_ek_akad                           Medianeinkommen akademischer Berufsabschluss
     m_mietpr                                                     Angebotsmietpreise
        q_alo                                                      Arbeitslosenquote
     q_bip_ew                                      Bruttoinlandsprodukt je Einwohner
        q_svw                                                    

In [45]:
missing_after_extraction = (
    set(SELECTED_CODES)
    - set(state_raw["Kuerzel"].unique())
)

print("Missing indicators:", missing_after_extraction)

Missing indicators: set()


In [46]:
RAW_STATE_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "inkar_bundeslaender_raw.csv"
)

RAW_STATE_PARQUET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "inkar_bundeslaender_raw.parquet"
)

state_raw.to_csv(
    RAW_STATE_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

state_raw.to_parquet(
    RAW_STATE_PARQUET_PATH,
    index=False
)

print("CSV saved:", RAW_STATE_CSV_PATH)
print("Parquet saved:", RAW_STATE_PARQUET_PATH)
print("CSV exists:", RAW_STATE_CSV_PATH.exists())
print("Parquet exists:", RAW_STATE_PARQUET_PATH.exists())

CSV saved: C:\Users\maazm\Desktop\Projects\datacareer-germany\data\processed\inkar_bundeslaender_raw.csv
Parquet saved: C:\Users\maazm\Desktop\Projects\datacareer-germany\data\processed\inkar_bundeslaender_raw.parquet
CSV exists: True
Parquet exists: True


In [47]:
print("Sample values:")
print(state_raw["Wert"].dropna().drop_duplicates().head(30).tolist())

Sample values:
['75,61', '89,94', '90,31', '92,83', '95,32', '9,58', '9,66', '9,81', '10,07', '10,20', '10,32', '10,50', '10,71', '10,90', '11,13', '10,86', '2,53', '2,58', '2,61', '2,67', '2,70', '2,73', '2,82', '2,91', '3,00', '3,09', '3,23', '7,40', '7,28', '7,06']


In [48]:
non_numeric_values = state_raw[
    ~state_raw["Wert"]
    .fillna("")
    .str.strip()
    .str.match(r"^-?\d+([,.]\d+)?$")
]["Wert"].value_counts(dropna=False)

display(non_numeric_values.head(20))

Series([], Name: count, dtype: int64)

In [49]:
state_clean = state_raw.copy()

state_clean = state_clean.rename(
    columns={
        "Kennziffer": "state_code",
        "Name": "state_name",
        "Zeitbezug": "year",
        "Wert": "value",
        "Kuerzel": "indicator_code",
        "Indikator": "indicator_name"
    }
)

# Preserve leading zeros in German state codes
state_clean["state_code"] = (
    state_clean["state_code"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

# Convert year to an integer
state_clean["year"] = pd.to_numeric(
    state_clean["year"],
    errors="coerce"
).astype("Int64")

# Convert German numeric format: 7,50 -> 7.50
state_clean["value"] = (
    state_clean["value"]
    .astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

state_clean["value"] = pd.to_numeric(
    state_clean["value"],
    errors="coerce"
)

# Add our professional English metric names
state_clean["metric"] = state_clean[
    "indicator_code"
].map(SELECTED_CODES)

state_clean = state_clean[
    [
        "state_code",
        "state_name",
        "year",
        "indicator_code",
        "indicator_name",
        "metric",
        "value"
    ]
].copy()

display(state_clean.head(20))

,state_code,state_name,year,indicator_code,indicator_name,metric,value
0,01,Schleswig-Holstein,2017,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,broadband_100mbit,75.61
1,01,Schleswig-Holstein,2020,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,broadband_100mbit,89.94
2,01,Schleswig-Holstein,2021,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,broadband_100mbit,90.31
3,01,Schleswig-Holstein,2022,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,broadband_100mbit,92.83
4,01,Schleswig-Holstein,2023,a_bb_100Mbits,Bandbreitenverfügbarkeit mindestens 100 Mbit/s,broadband_100mbit,95.32
5,01,Schleswig-Holstein,2013,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,expert_level_employment,9.58
6,01,Schleswig-Holstein,2014,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,expert_level_employment,9.66
7,01,Schleswig-Holstein,2015,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,expert_level_employment,9.81
8,01,Schleswig-Holstein,2016,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,expert_level_employment,10.07
9,01,Schleswig-Holstein,2017,a_sva_exp,Beschäftigte mit Anforderungsniveau Experte,expert_level_employment,10.20


In [50]:
print("Cleaned shape:", state_clean.shape)
print("Number of states:", state_clean["state_code"].nunique())
print("Number of metrics:", state_clean["metric"].nunique())
print("Earliest year:", state_clean["year"].min())
print("Latest year:", state_clean["year"].max())

print("\nMissing values:")
print(
    state_clean[
        [
            "state_code",
            "state_name",
            "year",
            "metric",
            "value"
        ]
    ].isna().sum()
)

print("\nData types:")
print(state_clean.dtypes)

Cleaned shape: (3760, 7)
Number of states: 16
Number of metrics: 12
Earliest year: 1995
Latest year: 2024

Missing values:
state_code    0
state_name    0
year          0
metric        0
value         0
dtype: int64

Data types:
state_code            str
state_name            str
year                Int64
indicator_code        str
indicator_name        str
metric                str
value             float64
dtype: object


In [51]:
duplicate_mask = state_clean.duplicated(
    subset=[
        "state_code",
        "year",
        "metric"
    ],
    keep=False
)

duplicate_rows = state_clean[
    duplicate_mask
].sort_values(
    [
        "state_code",
        "year",
        "metric"
    ]
)

print("Number of duplicate rows:", len(duplicate_rows))

display(duplicate_rows.head(20))

Number of duplicate rows: 0


,state_code,state_name,year,indicator_code,indicator_name,metric,value


In [52]:
if len(duplicate_rows) > 0:
    raise ValueError(
        "Duplicate state-year-metric observations found. "
        "Inspect duplicate_rows before continuing."
    )

print("Duplicate check passed.")

Duplicate check passed.


In [53]:
state_year = (
    state_clean
    .pivot(
        index=[
            "state_code",
            "state_name",
            "year"
        ],
        columns="metric",
        values="value"
    )
    .reset_index()
)

state_year.columns.name = None

metric_columns = list(SELECTED_CODES.values())

state_year = state_year[
    [
        "state_code",
        "state_name",
        "year",
        *metric_columns
    ]
].sort_values(
    [
        "state_code",
        "year"
    ]
).reset_index(drop=True)

print("Wide dataset shape:", state_year.shape)
print("Number of states:", state_year["state_code"].nunique())
print("Years:", state_year["year"].min(), "to", state_year["year"].max())

display(state_year.head(20))

Wide dataset shape: (480, 15)
Number of states: 16
Years: 1995 to 2024


,state_code,state_name,year,population,unemployment_rate,asking_rent,employment_rate,expert_level_employment,knowledge_intensive_industry,it_science_service_employment,population_change_10_years,total_migration_balance,median_income_academic,broadband_100mbit,gdp_per_capita
0,01,Schleswig-Holstein,1995,2725461.0,NaN,NaN,NaN,NaN,NaN,NaN,4.26,7.68,NaN,NaN,21.18
1,01,Schleswig-Holstein,1996,2742293.0,NaN,NaN,NaN,NaN,NaN,NaN,4.96,7.07,NaN,NaN,21.42
2,01,Schleswig-Holstein,1997,2756473.0,NaN,NaN,47.20,NaN,NaN,NaN,5.49,5.58,NaN,NaN,21.80
3,01,Schleswig-Holstein,1998,2766057.0,9.97,NaN,46.76,NaN,NaN,NaN,7.86,4.30,NaN,NaN,22.03
4,01,Schleswig-Holstein,1999,2777275.0,9.39,NaN,47.51,NaN,NaN,NaN,7.04,5.03,NaN,NaN,22.18
5,01,Schleswig-Holstein,2000,2789761.0,8.51,NaN,48.03,NaN,NaN,NaN,6.23,5.52,NaN,NaN,22.85
6,01,Schleswig-Holstein,2001,2804249.0,8.45,NaN,47.91,NaN,NaN,NaN,5.88,6.59,NaN,NaN,23.41
7,01,Schleswig-Holstein,2002,2816507.0,8.71,NaN,47.63,NaN,NaN,NaN,5.11,6.12,NaN,NaN,23.00
8,01,Schleswig-Holstein,2003,2823171.0,9.71,NaN,46.63,NaN,NaN,NaN,4.76,4.60,NaN,NaN,23.19
9,01,Schleswig-Holstein,2004,2828760.0,9.84,NaN,46.06,NaN,NaN,NaN,4.44,3.94,NaN,NaN,23.65


In [54]:
metric_coverage = pd.DataFrame(
    {
        "non_missing_rows": state_year[
            metric_columns
        ].notna().sum(),

        "missing_rows": state_year[
            metric_columns
        ].isna().sum(),

        "first_year": [
            state_year.loc[
                state_year[column].notna(),
                "year"
            ].min()
            for column in metric_columns
        ],

        "latest_year": [
            state_year.loc[
                state_year[column].notna(),
                "year"
            ].max()
            for column in metric_columns
        ]
    }
).reset_index(
    names="metric"
)

display(metric_coverage)

,metric,non_missing_rows,missing_rows,first_year,latest_year
0,population,464,16,1995,2023
1,unemployment_rate,416,64,1998,2023
2,asking_rent,240,240,2010,2024
3,employment_rate,432,48,1997,2023
4,expert_level_employment,176,304,2013,2023
5,knowledge_intensive_industry,240,240,2009,2023
6,it_science_service_employment,176,304,2013,2023
7,population_change_10_years,464,16,1995,2023
8,total_migration_balance,464,16,1995,2023
9,median_income_academic,160,320,2014,2023


In [55]:
CLEAN_LONG_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "inkar_bundeslaender_clean_long.parquet"
)

STATE_YEAR_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "germany_state_year.csv"
)

STATE_YEAR_PARQUET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "germany_state_year.parquet"
)

state_clean.to_parquet(
    CLEAN_LONG_PATH,
    index=False
)

state_year.to_csv(
    STATE_YEAR_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

state_year.to_parquet(
    STATE_YEAR_PARQUET_PATH,
    index=False
)

print("Long dataset saved:", CLEAN_LONG_PATH.exists())
print("Wide CSV saved:", STATE_YEAR_CSV_PATH.exists())
print("Wide Parquet saved:", STATE_YEAR_PARQUET_PATH.exists())

Long dataset saved: True
Wide CSV saved: True
Wide Parquet saved: True


In [56]:
connection.close()
print("DuckDB connection closed.")

DuckDB connection closed.
